# Challenge 5a: Classifying Sea State from Buoy Observations

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Preprocessing](#44-preprocessing)
5. [The Challenge](#5-the-challenge)
   - [5.1 Training the Model](#51-training-the-model)
   - [5.2 Evaluating the Model](#52-evaluating-the-model)
   - [5.3 Interpretation](#53-interpretation)
   - [5.4 Reflection Questions](#54-reflection-questions)
6. [Solution](#6-solution)
7. [References](#7-references)

---
## 1. Introduction

Knowing whether the sea is calm or rough is not a casual observation. It
determines whether a cargo vessel can maintain schedule, whether an offshore
wind turbine maintenance crew can transfer safely to the platform, whether a
search-and-rescue helicopter can operate, and whether a coastal flood warning
needs to be issued. Operational oceanographers make these calls continuously,
drawing on data from moored buoys, satellite altimeters, and numerical
weather models. Automating the classification from sensor measurements is a
practical problem with direct safety consequences.

Sea state is defined formally by the World Meteorological Organisation (WMO)
using a nine-point scale keyed to significant wave height, the average height
of the highest one-third of waves in a record. In practice, the boundary
between operationally safe and hazardous conditions is typically set at
significant wave heights of 2 to 2.5 metres, corresponding roughly to
Beaufort force 5-6 and WMO sea state codes 4-5. Below this threshold,
conditions are workable for most marine operations. Above it, many operations
are restricted or suspended.

In this challenge you will train a binary logistic regression classifier to
distinguish calm sea conditions (WMO states 0-3, Beaufort 0-4) from storm
conditions (WMO states 5-9, Beaufort 7+) using five variables routinely
measured by oceanographic buoys: wind speed, dominant wave period, significant
swell height, sea surface temperature anomaly, and mean sea level pressure.
The first four features carry most of the signal. Pressure is negatively
correlated with wind and wave variables because low-pressure depressions drive
storm winds, and this correlation is something you should observe and reason
about. The SST anomaly is the weakest predictor: it reflects longer-term ocean
state rather than the instantaneous meteorological forcing, and it introduces
useful noise into the problem.

This is intentionally an entry-level challenge. The dataset is balanced,
the signal is reasonably clean, and standard logistic regression without
modification should achieve good performance. The focus is on doing the
basics correctly: scaling the features, choosing the right evaluation
metrics, and understanding what the fitted model is telling you.

### Learning Objectives

- Apply binary logistic regression to a real-world classification problem
  with continuous meteorological features
- Recognise why feature scaling is necessary before fitting logistic
  regression with a gradient-based solver, and apply it without data leakage
- Evaluate a binary classifier using accuracy, precision, recall, F1 score,
  and the confusion matrix, and interpret each metric in the context of
  the problem
- Inspect the fitted coefficients of a logistic regression model and
  reason about their signs and magnitudes using domain knowledge
- Identify which features contribute most to the classification decision
  and which contribute least

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `LogisticRegression` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html |
| `StandardScaler` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html |
| `classification_report` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| `ConfusionMatrixDisplay` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html |
| WMO Sea State Code table | https://www.nodc.noaa.gov/General/NODC-Archive/seastate.html |
| NOAA National Data Buoy Center | https://www.ndbc.noaa.gov |
| Significant wave height definition (ECMWF) | https://www.ecmwf.int/en/forecasts/documentation-and-support/waves/wave-parameters |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later. All libraries are part of the
standard scientific Python stack available in any Anaconda or pip-based
environment that includes scikit-learn.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting the dataset |
| `matplotlib` | Plotting |
| `seaborn` | Distribution plots and heatmaps |
| `sklearn.model_selection` | Stratified train/test splitting |
| `sklearn.preprocessing` | Feature scaling via `StandardScaler` |
| `sklearn.linear_model` | `LogisticRegression` classifier |
| `sklearn.metrics` | Confusion matrix, classification report, accuracy |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `sea_state.csv` |
| Location | `data/sea_state.csv` (relative to this notebook) |
| Total rows | 2,500 |
| Features | 5 continuous columns |
| Target column | `sea_state` |
| Class 0 | calm: 1,237 samples (~49.5%) |
| Class 1 | storm: 1,263 samples (~50.5%) |

The dataset is balanced. Both classes contribute approximately equally to
the training and test sets, so accuracy is a reasonable headline metric
here, unlike in Challenge 5 where the majority class dominated.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/sea_state.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(10)

### 4.2 Understanding the Features

All five features are quantities routinely recorded by moored oceanographic
buoys operated by agencies such as NOAA, the UK Met Office, and the Irish
Marine Institute. The values in this dataset are synthetic but the
distributions and physical relationships between variables are realistic.

| Feature | Type | Units | Description |
|---|---|---|---|
| `wind_speed_ms` | Continuous | m/s | Wind speed measured at 10 metres above sea level. The primary driver of wave growth in deep water. Calm conditions: typically below 7 m/s. Storm conditions: commonly above 14 m/s. |
| `wave_period_s` | Continuous | s | Dominant wave period: the time between successive crests of the most energetic wave component. Calm seas have short, choppy periods (4-8 s). Long-period swell (>12 s) indicates active storm forcing or swell arriving from a distant storm. |
| `swell_height_m` | Continuous | m | Significant swell height: the mean height of the highest one-third of waves. This is the standard WMO definition of wave height. Calm: typically below 1.5 m. Storm: typically above 2.5 m. |
| `sst_anomaly_c` | Continuous | deg C | Sea surface temperature anomaly relative to the climatological mean for the location and month. A positive anomaly can amplify storm intensity via increased evaporation. Its correlation with instantaneous sea state is real but weak. |
| `pressure_hpa` | Continuous | hPa | Mean sea level atmospheric pressure. Low-pressure depressions drive storm winds. Calm conditions are associated with high-pressure ridges. A rule of thumb: pressure below 1000 hPa at a buoy often indicates nearby storm forcing. |
| `sea_state` | Binary | label | Target variable. 0 = calm (WMO sea state 0-3). 1 = storm (WMO sea state 5-9). |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['sea_state'].value_counts().sort_index()
names = {0: 'calm', 1: 'storm'}
for label, count in counts.items():
    print(f'  Class {label} ({names[label]:6s}): {count:5d}  ({100*count/len(df):.1f}%)')

print()
print('Mean feature values by class:')
print(df.groupby('sea_state')[
    ['wind_speed_ms', 'wave_period_s', 'swell_height_m', 'sst_anomaly_c', 'pressure_hpa']
].mean().round(2).to_string())

**Questions to consider:**

- The class means are very different for wind speed, wave period, and swell
  height. Based on this alone, do you expect logistic regression to find
  these features easy or hard to use?
- The SST anomaly means are almost identical between the two classes. What
  does this tell you about how much this feature will contribute to the
  classification? Does that match the physical description in Section 4.2?
- Pressure is lower on average for storm conditions, as expected. Is the
  difference as large as for the wave-related features?

In [ ]:
# Listing 4.
# Feature distributions by class
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
features = ['wind_speed_ms', 'wave_period_s', 'swell_height_m', 'sst_anomaly_c', 'pressure_hpa']
colours  = {0: 'steelblue', 1: 'firebrick'}

for ax, feat in zip(axes, features):
    for cls in [0, 1]:
        subset = df.loc[df['sea_state'] == cls, feat]
        subset.plot(kind='kde', ax=ax, label=names[cls],
                    color=colours[cls], linewidth=2)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions by Sea State', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Questions to consider:**

- Which features show the cleanest separation between calm and storm?
  Do any features show almost no separation at all?
- The calm and storm distributions for `swell_height_m` and `wind_speed_ms`
  are widely separated and non-overlapping. What does this suggest about
  how confident the classifier should be for most predictions?
- Where the distributions do overlap (look at `pressure_hpa`), what does
  this tell you about the cases the model is likely to get wrong?

In [ ]:
# Listing 5.
# Correlation matrix
corr = df.drop(columns='sea_state').corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True
)
ax.set_title('Feature Correlation Matrix', fontsize=12)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `wind_speed_ms`, `wave_period_s`, and `swell_height_m` are strongly
  correlated with each other. Can you explain this physically? (Hint: wave
  growth is driven by wind speed and fetch duration.)
- `pressure_hpa` is negatively correlated with the three wave-related
  features. Why would this be? What does the sign of the correlation tell
  you?
- `sst_anomaly_c` is almost uncorrelated with everything else. Is this a
  problem for logistic regression, or does it just mean this feature will
  receive a small coefficient?
- Given the high correlation among the three wave features, do you think
  the model needs all three, or could one of them be removed without much
  loss? How would you test this?

### 4.4 Preprocessing

As covered in Notebook_5, logistic regression uses a gradient-based solver
(`lbfgs` by default) to find the optimal coefficients. The solver is
sensitive to the scale of the input features. Here, `pressure_hpa` ranges
from roughly 940 to 1040 hPa, while `sst_anomaly_c` ranges from about
-3 to +3 degrees. Without scaling, the gradient with respect to pressure
is orders of magnitude smaller than the gradient with respect to SST
anomaly, which can cause the solver to converge slowly or to a suboptimal
solution.

`StandardScaler` transforms each feature to zero mean and unit variance.
After scaling, all five features contribute equally to the gradient
computation, and the solver converges quickly.

The rule to remember: fit the scaler on the training data, then use
those same parameters (mean and standard deviation per feature) to
transform the test data. Never fit on the test set or on the combined
dataset. Doing so would allow test-set statistics to influence the
training process, inflating your reported performance.

The dataset is balanced and fairly small (2,500 samples). A 75/25 split
gives 625 test samples, which is enough for stable binary metrics.
Stratification is good practice even with a balanced dataset.

In [ ]:
# Listing 6.
# TODO: Implement preprocessing.
#
# 1. Separate the features (X) from the target column ('sea_state') (y).
#    Use all five feature columns.
#
# 2. Split X and y into training and test sets.
#    Use a 75/25 split, random_state=42, and stratify by y.
#
# 3. Apply StandardScaler to the features.
#    Fit the scaler on X_train only, then transform both X_train and X_test.
#
# 4. Print the shapes of X_train, X_test, y_train, y_test after splitting.
#    Also print the class counts in y_train and y_test to confirm that
#    stratification has preserved the 50/50 balance.

# YOUR CODE HERE

---
## 5. The Challenge

Your task is to train a binary logistic regression classifier that predicts
whether sea conditions are calm or stormy from the five buoy measurements.

This is a binary classification problem, so `LogisticRegression` uses the
standard sigmoid function rather than the softmax used in Challenge 5.
The output is a probability that the observation belongs to class 1 (storm).
The default decision threshold is 0.5: predict storm if P(storm) > 0.5,
calm otherwise.

### 5.1 Training the Model

In [ ]:
# Listing 7.
# TODO: Train the logistic regression model.
#
# 1. Instantiate LogisticRegression.
#    Suggested starting parameters:
#      - solver='lbfgs'  (default; handles binary problems well)
#      - C=1.0           (default regularisation strength; try other values later)
#      - max_iter=200    (should be more than enough for scaled data)
#      - random_state=42
#
# 2. Fit the model on the scaled training data.
#
# 3. After fitting, print:
#      - How many iterations the solver took (.n_iter_)
#      - The shape of .coef_  (should be (1, 5) for binary classification)
#      - The intercept (.intercept_)

# YOUR CODE HERE

### 5.2 Evaluating the Model

With a balanced dataset, accuracy is a fair headline metric: a classifier
that always predicted the majority class would only achieve ~50%, so
improvements above that are genuinely meaningful. That said, accuracy
alone does not tell you where the model succeeds and where it fails.

Use the full classification report to check precision, recall, and F1 for
both classes. For an operational sea state system:

- **Recall for storm (class 1)** matters most. Missing a storm (false
  negative) means issuing no warning when one was needed. This is the
  more costly error in a safety context.
- **Precision for storm (class 1)** reflects how often a storm warning
  turns out to be correct. Low precision means frequent false alarms,
  which erode trust in the system.

The confusion matrix makes both types of error visible at once.

In [ ]:
# Listing 8.
# TODO: Evaluate the model.
#
# 1. Generate predictions on the scaled test set.
#
# 2. Print the overall accuracy.
#
# 3. Print the full classification report with
#    target_names=['calm', 'storm'].
#
# 4. Plot the confusion matrix using ConfusionMatrixDisplay.from_predictions().
#    Use display_labels=['calm', 'storm'].
#    Look at the off-diagonal cells: which error type (false calm or false
#    storm) does the model make more often?

# YOUR CODE HERE

### 5.3 Interpretation

The fitted `LogisticRegression` model stores a coefficient for each feature
in `clf.coef_`, which has shape `(1, n_features)` for binary classification.
After scaling, the coefficient magnitudes are directly comparable: a larger
absolute value means the model assigns more weight to that feature when
computing the log-odds of storm vs calm.

Positive coefficients increase the probability of storm as the feature
increases. Negative coefficients decrease it. Check whether the signs
match physical intuition: wind speed and wave height should push toward
storm (positive), while pressure should push away from it (negative).

In [ ]:
# Listing 9.
# TODO: Interpret the coefficients.
#
# 1. Create a pandas Series from clf.coef_[0] with the feature names as
#    the index. Print it.
#
# 2. Create a horizontal bar chart of the coefficients, sorted by magnitude.
#    Use a diverging colour scheme so positive and negative coefficients
#    are visually distinct (e.g. red for positive, blue for negative).
#    Label the x-axis 'Coefficient (log-odds of storm)'.
#
# 3. Look at the SST anomaly coefficient. Is it near zero, as the exploratory
#    analysis suggested it would be?
#
# 4. The three wave-related features are strongly intercorrelated. Does the
#    model spread weight roughly equally across them, or does it concentrate
#    on one? What might explain this?

# YOUR CODE HERE

### 5.4 Reflection Questions

1. The model predicts a probability rather than a hard label. The default
   decision threshold is 0.5. If you were deploying this model as a storm
   warning system and missing a storm was ten times worse than a false alarm,
   how would you adjust the threshold? What effect would this have on
   precision and recall?

2. The three wave features are highly correlated. Would removing two of them
   and keeping only `swell_height_m` change the classification performance
   meaningfully? How would you test this cleanly?

3. Logistic regression fits a linear decision boundary in feature space.
   Given what you saw in the KDE plots, do you think a linear boundary is
   adequate for this problem, or do the distributions suggest the boundary
   is already approximately linear?

4. The dataset contains about 3% label noise, reflecting real ambiguity at
   the calm/storm boundary. What is the theoretical upper limit on accuracy
   for a perfect classifier given this noise level? Is the model approaching
   that limit?

---
## 6. Solution

**Read this section only after attempting the challenge yourself.**
Every code cell below is explained with the reasoning behind each decision,
not just a description of what the code does. The cells are self-contained
and can be run from top to bottom without any variables from Section 5.

### Step 1: Preprocessing

We use a 75/25 split. With 2,500 samples this gives 1,875 training examples
and 625 test examples. The test set is large enough that binary metrics
(precision, recall, F1) will be stable. With a perfectly balanced dataset
stratification makes almost no numerical difference, but it is good practice:
if the random split happened to give 55% storm in training and 45% in test,
the reported accuracy would be slightly misleading.

Scaling is important here. The variable `pressure_hpa` has a mean around 1004 and
a standard deviation of about 15. `sst_anomaly_c` has a mean near zero and
a standard deviation of about 1.2. If you give these raw to `lbfgs`, the
pressure gradient will be roughly ten times smaller than the SST gradient
per unit change, causing the solver to underfit the pressure feature.
After `StandardScaler`, both features have mean 0 and standard deviation 1,
and the solver treats them symmetrically.

In [ ]:
# Listing 10.
feature_cols = ['wind_speed_ms', 'wave_period_s', 'swell_height_m',
                'sst_anomaly_c', 'pressure_hpa']
X = df[feature_cols].values
y = df['sea_state'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set:  {X_train_sc.shape[0]} samples')
print(f'Test set:      {X_test_sc.shape[0]} samples')
print()
print('Class counts in training set:')
for cls in [0, 1]:
    n = np.sum(y_train == cls)
    print(f'  {names[cls]:6s}: {n} ({100*n/len(y_train):.1f}%)')
print('Class counts in test set:')
for cls in [0, 1]:
    n = np.sum(y_test == cls)
    print(f'  {names[cls]:6s}: {n} ({100*n/len(y_test):.1f}%)')

### Step 2: Training the Model

There is nothing exotic to configure here. Binary logistic regression with
`lbfgs` and `C=1.0` is the appropriate starting point for a problem where
the classes are well-separated, the dataset is small and balanced, and no
prior knowledge suggests the default regularisation strength is wrong.

`max_iter=200` is generous for five scaled features. In practice, `lbfgs`
converges in fewer than 30 iterations on this dataset.

In [ ]:
# Listing 11.
clf = LogisticRegression(
    solver='lbfgs',
    C=1.0,
    max_iter=200,
    random_state=42
)
clf.fit(X_train_sc, y_train)

print(f'Solver converged in {clf.n_iter_[0]} iterations.')
print(f'Coefficient shape: {clf.coef_.shape}  (1 row for binary classification)')
print(f'Intercept: {clf.intercept_[0]:.4f}')

The coefficient matrix has shape (1, 5) for a binary problem: one row
containing a weight for each of the five features. The intercept reflects
the baseline log-odds when all features are at their mean (zero after
scaling). An intercept near zero is expected when the classes are balanced
and the feature means are similar between classes.

### Step 3: Evaluating the Model

Because the classes are balanced, accuracy gives a meaningful picture here.
A classifier that predicted "storm" for everything would score 50%; anything
meaningfully above that reflects genuine learning. Still, the classification
report breaks the headline number down by class, which shows whether the
model is performing symmetrically or favouring one class.

In [ ]:
# Listing 12.
y_pred = clf.predict(X_test_sc)

acc = accuracy_score(y_test, y_pred)
print(f'Overall accuracy: {acc:.4f}\n')
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=['calm', 'storm']))

With clean, well-separated features, logistic regression achieves high
accuracy on this problem. The SST anomaly feature is weak, but the three
wave features together provide a near-complete signal, and pressure adds
complementary information. The main source of error is the ~3% label noise
in the dataset, which sets an approximate ceiling on achievable performance
regardless of the algorithm.

Check whether precision and recall are symmetric between the two classes.
If one class has substantially lower recall, it means the model is
systematically missing that class. With a balanced dataset and clean
features, this should not be a major problem here.

In [ ]:
# Listing 13.
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['calm', 'storm'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix: Sea State Classification\n(LogisticRegression, binary)', fontsize=10)
plt.tight_layout()
plt.show()

The confusion matrix shows the raw counts of correct and incorrect
predictions in each cell. The diagonal holds the correct predictions.
Off-diagonal cells are misclassifications: calm predicted as storm (top
right) and storm predicted as calm (bottom left). For a storm warning
system, the bottom-left cell (missed storms) is the operationally
critical one.

### Step 4: Interpreting the Coefficients

After scaling, the coefficients are on a common scale and can be compared
directly. A coefficient of 2.5 for `wind_speed_ms` means that a
one-standard-deviation increase in wind speed increases the log-odds of
storm by 2.5 units, which corresponds to multiplying the odds by
exp(2.5) ≈ 12. This is a large effect.

The SST anomaly coefficient is expected to be small in magnitude. High
intercorrelation among wind speed, wave period, and swell height means the
model may concentrate weight on whichever of the three the solver finds most
numerically convenient, while still producing correct predictions. This is
a known behaviour of regularised linear models with collinear inputs: the
coefficient values become less individually interpretable, even though the
predictions remain accurate.

In [ ]:
# Listing 14.
coef_series = pd.Series(clf.coef_[0], index=feature_cols)
print('Logistic regression coefficients (log-odds of storm per unit scaled feature):')
print(coef_series.round(4).to_string())
print()

# Horizontal bar chart sorted by magnitude
coef_sorted = coef_series.reindex(coef_series.abs().sort_values().index)
colours_bar = ['firebrick' if c > 0 else 'steelblue' for c in coef_sorted]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(coef_sorted.index, coef_sorted.values, color=colours_bar)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (log-odds of storm)', fontsize=11)
ax.set_title('Logistic Regression Coefficients', fontsize=12)
plt.tight_layout()
plt.show()

**Reading the chart:**

- Red bars (positive coefficients) increase the probability of storm as
  the feature increases. Wind speed, wave period, and swell height should
  all be red: higher values indicate rougher conditions.
- Blue bars (negative coefficients) decrease the probability of storm.
  Pressure should be blue: higher pressure is associated with calmer
  conditions.
- The SST anomaly bar should be short in both directions, confirming it
  contributes little to the decision.
- Among the three correlated wave features, the model may assign unequal
  weights depending on which combination the L2 regulariser found most
  compact. This does not indicate one feature is more physically important
  than another; it reflects the redundancy among correlated inputs.

---
## 7. References

1. Cox, D.R. (1958). The regression analysis of binary responses.
   *Journal of the Royal Statistical Society, Series B*, 20(2), 215-242.
   The foundational paper for logistic regression.

2. World Meteorological Organisation. (2018). *Guide to Wave Analysis and
   Forecasting*. WMO-No. 702. WMO, Geneva.
   https://library.wmo.int/index.php?lvl=notice_display&id=223
   Authoritative reference for sea state definitions, significant wave
   height, and the WMO sea state code.

3. NOAA National Data Buoy Center. Standard Meteorological Data.
   https://www.ndbc.noaa.gov/measdes.shtml
   Documentation for the buoy variables that inspired the features in
   this dataset, including wind speed, wave period, and wave height.

4. Holthuijsen, L.H. (2007). *Waves in Oceanic and Coastal Waters*.
   Cambridge University Press.
   Chapter 3 covers the physical relationship between wind speed, fetch,
   wave period, and significant wave height.

5. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html
   Cite this when reporting results obtained with scikit-learn.